# Notebook 8 — Final Model Selection and Validation Summary

## Purpose
This notebook consolidates the full modeling validation into one explanatory, presentation-ready notebook.

It explains:
- the benchmark setup
- the validated modeling pipeline
- the model candidates and their architecture
- the selected winners for each branch
- the key results and figures for the validation presentation

## Current validated benchmark
- real data only
- current supervised benchmark classes:
  - `RC_CAPACITY_OVERLOAD`
  - `RC_TRANSPORT_DELAY`
- exploratory rare class:
  - `RC_CQI_MISMATCH`
- main temporal benchmark window:
  - **10 samples**

## What this notebook is for

Use this notebook as the final modeling-validation summary for your presentation.

It contains:
1. the benchmark scope
2. the validated hybrid architecture
3. explanation of the supervised models
4. explanation of the temporal autoencoders and their layers
5. explanation of the prototype methods
6. the final winners:
   - supervised branch
   - temporal branch
   - prototype branch
7. presentation-ready Plotly figures
8. a final conclusion and future-work section

In [6]:
import os
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

VERBOSE = True
EXPORT_PLOTS = True

def log(msg, level="INFO"):
    if VERBOSE:
        print(f"[{level}] {msg}")

def banner(title):
    if VERBOSE:
        print("\n" + "=" * 90)
        print(title)
        print("=" * 90)

banner("NOTEBOOK 8 INITIALIZATION")
log("Notebook 8 initialized")


NOTEBOOK 8 INITIALIZATION
[INFO] Notebook 8 initialized


In [7]:
banner("INSTALL / IMPORT VISUALIZATION PACKAGES")

try:
    import plotly.express as px
    import plotly.graph_objects as go
except Exception:
    import sys
    !{sys.executable} -m pip install -q plotly
    import plotly.express as px
    import plotly.graph_objects as go

try:
    import kaleido  # noqa: F401
    log("kaleido already available")
except Exception:
    log("Installing kaleido for Plotly PNG export...", "WARN")
    import sys
    !{sys.executable} -m pip install -q kaleido
    import kaleido  # noqa: F401
    log("kaleido installed successfully")

def save_plot(fig, name):
    if not EXPORT_PLOTS:
        return
    fig.write_html(f"{name}.html")
    try:
        fig.write_image(f"{name}.png", scale=2)
        log(f"Saved {name}.html and {name}.png")
    except Exception as e:
        log(f"Saved {name}.html | PNG export skipped ({e})", "WARN")


INSTALL / IMPORT VISUALIZATION PACKAGES
[INFO] kaleido already available


## Load the summary artifacts from the previous notebooks

We load:
- Notebook 4 supervised-branch summary
- Notebook 6 temporal-branch summary
- Notebook 7 prototype-branch summary

In [8]:
banner("LOAD PREVIOUS SUMMARIES")

with open("/kaggle/input/notebooks/azizamriiiiiiiiii/04-root-cause-models-compare/nb4_model_selection_summary_patched.json", "r") as f:
    nb4_summary = json.load(f)

with open("/kaggle/input/notebooks/azizamriiiiiiiiii/06-temporal-models-compare/nb6_temporal_model_summary.json", "r") as f:
    nb6_summary = json.load(f)

with open("/kaggle/input/notebooks/azizamriiiiiiiiii/notebook7/nb7_prototype_summary.json", "r") as f:
    nb7_summary = json.load(f)

nb4_metrics = pd.read_csv("/kaggle/input/notebooks/azizamriiiiiiiiii/04-root-cause-models-compare/nb4_model_metrics_patched.csv")
nb6_metrics = pd.read_csv("/kaggle/input/notebooks/azizamriiiiiiiiii/06-temporal-models-compare/nb6_temporal_model_comparison.csv")
nb7_metrics = pd.read_csv("/kaggle/input/notebooks/azizamriiiiiiiiii/notebook7/nb7_prototype_metrics.csv")

display(pd.DataFrame([{
    "supervised_winner": nb4_summary["best_model"],
    "temporal_winner": nb6_summary["best_temporal_model"],
    "prototype_winner": nb7_summary["best_prototype_method"],
    "target_classes": ", ".join(nb4_summary["target_classes"]),
    "window_size": nb6_summary["window_size"],
}]))


LOAD PREVIOUS SUMMARIES


,supervised_winner,temporal_winner,prototype_winner,target_classes,window_size
0,Random Forest,GRU,FAISS,"RC_CAPACITY_OVERLOAD, RC_TRANSPORT_DELAY",10


# 1. Benchmark definition

## Current benchmark scope
The current benchmark is intentionally limited to the root causes that the real-data labeling pipeline supported robustly enough for supervised validation:

- `RC_CAPACITY_OVERLOAD`
- `RC_TRANSPORT_DELAY`

A small number of `RC_CQI_MISMATCH` examples appeared in the all-data labeling stage, but they were too underrepresented to become a stable supervised class in this validation.

## Why this is acceptable
The final Diagnostic Agent architecture targets a broader taxonomy, but the current validation focuses on the classes that are most defensible with real data only.

In [9]:
banner("BENCHMARK SNAPSHOT")

benchmark_snapshot = pd.DataFrame([{
    "target_classes": ", ".join(nb4_summary["target_classes"]),
    "positive_class_for_auc": nb4_summary["positive_class_for_roc_pr"],
    "n_features_after_patch": nb4_summary["n_features_after_patch"],
    "window_size": nb6_summary["window_size"],
    "supervised_winner": nb4_summary["best_model"],
    "temporal_winner": nb6_summary["best_temporal_model"],
    "prototype_winner": nb7_summary["best_prototype_method"],
}])

display(benchmark_snapshot)


BENCHMARK SNAPSHOT


,target_classes,positive_class_for_auc,n_features_after_patch,window_size,supervised_winner,temporal_winner,prototype_winner
0,"RC_CAPACITY_OVERLOAD, RC_TRANSPORT_DELAY",RC_TRANSPORT_DELAY,104,10,Random Forest,GRU,FAISS


# 2. Validated hybrid architecture

The current validated architecture for the benchmark can be summarized as:

### Branch A — Supervised tabular diagnosis
- input: current KPI snapshot + engineered features
- candidate models:
  - Random Forest
  - XGBoost
- selected winner:
  - **Random Forest**

### Branch B — Temporal representation
- input: 10-sample sequence windows
- candidate models:
  - LSTM Autoencoder
  - GRU Autoencoder
- selected winner:
  - **GRU Autoencoder**

### Branch C — Prototype retrieval
- input: GRU latent embeddings
- candidate methods:
  - NearestNeighbors
  - FAISS
- selected winner:
  - **FAISS**

This gives a coherent hybrid path:
- a supervised classifier for the benchmark diagnosis
- a temporal encoder for sequence representation
- a fast prototype backend for retrieval support

In [10]:
banner("PLOT VALIDATED HYBRID ARCHITECTURE")

# Build a simple Sankey diagram for the validated hybrid pipeline
labels = [
    "Real QoS Data",
    "Feature Engineering",
    "Random Forest",
    "Sequence Windows (w=10)",
    "GRU Autoencoder",
    "FAISS",
    "Final Hybrid Diagnostic Validation"
]

source = [0, 1, 0, 3, 4, 2, 5]
target = [1, 2, 3, 4, 5, 6, 6]
value  = [10, 10, 8, 8, 8, 7, 7]

fig_arch = go.Figure(data=[go.Sankey(
    node=dict(
        pad=18,
        thickness=18,
        line=dict(width=0.5),
        label=labels
    ),
    link=dict(
        source=source,
        target=target,
        value=value
    )
)])

fig_arch.update_layout(
    title_text="Validated Hybrid Modeling Pipeline",
    font_size=12
)
fig_arch.show()

save_plot(fig_arch, "nb8_validated_hybrid_architecture")


PLOT VALIDATED HYBRID ARCHITECTURE


[WARN] Saved nb8_validated_hybrid_architecture.html | PNG export skipped (
Image export using the "kaleido" engine requires the kaleido package,
which can be installed using pip:
    $ pip install -U kaleido
)


# 3. Supervised models and their architecture

## Random Forest
Random Forest is an ensemble of decision trees trained on bootstrapped subsets of the data.
Each tree sees:
- a random subset of rows
- a random subset of features

The final prediction is obtained by aggregating the predictions of all trees.

### Why it is useful here
- strong baseline for tabular diagnostic data
- handles non-linear interactions well
- robust to mixed feature scales
- easy to interpret with feature importance

## XGBoost
XGBoost is a gradient-boosted tree ensemble.
Instead of training trees independently like Random Forest, it builds trees sequentially so that each new tree corrects the residual errors of the previous ones.

### Why it is useful here
- often very strong on structured tabular data
- handles complex feature interactions
- good fit for engineered diagnostic KPIs

In [11]:
banner("SUPERVISED BRANCH RESULTS")

display(nb4_metrics)

# Keep only validation and test rows for a cleaner presentation figure
nb4_plot_df = nb4_metrics[nb4_metrics["split"].isin(["val", "test"])].copy()
nb4_long = nb4_plot_df.melt(
    id_vars=["model", "split"],
    value_vars=["accuracy", "balanced_accuracy", "macro_f1", "roc_auc", "pr_auc"],
    var_name="metric",
    value_name="value"
)

fig_nb4 = px.bar(
    nb4_long,
    x="metric",
    y="value",
    color="model",
    barmode="group",
    facet_col="split",
    title="Supervised Branch — Random Forest vs XGBoost"
)
fig_nb4.update_layout(template="plotly_white")
fig_nb4.show()

save_plot(fig_nb4, "nb8_supervised_branch_comparison")


SUPERVISED BRANCH RESULTS


,model,split,accuracy,balanced_accuracy,macro_f1,weighted_f1,macro_precision,macro_recall,roc_auc,pr_auc
0,Random Forest,train,0.995249,0.994894,0.994894,0.995249,0.994894,0.994894,0.999903,0.999833
1,Random Forest,val,0.972603,0.972222,0.972556,0.972572,0.974359,0.972222,0.987988,0.989815
2,Random Forest,test,0.987952,0.988095,0.987952,0.987952,0.988095,0.988095,0.988966,0.992584
3,XGBoost,train,0.992874,0.993015,0.992352,0.992879,0.991703,0.993015,0.999709,0.999531
4,XGBoost,val,0.972603,0.972222,0.972556,0.972572,0.974359,0.972222,0.980480,0.987387
5,XGBoost,test,0.987952,0.988095,0.987952,0.987952,0.988095,0.988095,0.987224,0.991815


[WARN] Saved nb8_supervised_branch_comparison.html | PNG export skipped (
Image export using the "kaleido" engine requires the kaleido package,
which can be installed using pip:
    $ pip install -U kaleido
)


## Supervised branch conclusion

The corrected benchmark selected **Random Forest** as the best supervised model.

Why?
- Random Forest matched XGBoost on the main classification metrics
- it slightly outperformed XGBoost on the validation tie-break metrics used in the notebook
- after removing risky/leaky features, the benchmark remained strong

In [12]:
banner("SUPERVISED WINNER SNAPSHOT")

supervised_snapshot = pd.DataFrame([{
    "winner": nb4_summary["best_model"],
    "rf_train_time_sec": nb4_summary["rf_train_time_sec"],
    "xgb_train_time_sec": nb4_summary["xgb_train_time_sec"],
    "n_features_after_patch": nb4_summary["n_features_after_patch"],
    "dropped_risky_features_count": len(nb4_summary["dropped_risky_features"])
}])

display(supervised_snapshot)


SUPERVISED WINNER SNAPSHOT


,winner,rf_train_time_sec,xgb_train_time_sec,n_features_after_patch,dropped_risky_features_count
0,Random Forest,0.654,0.299,104,14


# 4. Temporal models and their architecture

## LSTM Autoencoder architecture
The LSTM autoencoder used in the temporal benchmark has the following structure:

1. **Input sequence**
   - shape: `(window_size, n_features)`
   - here the main window size is 10

2. **LSTM encoder**
   - compresses the whole sequence into a hidden state

3. **Linear latent layer**
   - projects the encoder state into a compact latent vector

4. **Linear expansion layer**
   - maps the latent vector back to a hidden representation

5. **Repeated latent sequence**
   - the expanded latent representation is repeated across time steps

6. **LSTM decoder**
   - reconstructs the sequence dynamics

7. **Output linear layer**
   - maps decoder outputs back to the original feature dimension

## GRU Autoencoder architecture
The GRU autoencoder follows the same overall design, but replaces the LSTM recurrent layers with **GRU** layers.

### Why compare LSTM and GRU?
- both are strong recurrent sequence models
- GRU is usually simpler and faster
- LSTM can sometimes model longer dependencies more richly
- the benchmark determines which one reconstructs the QoS windows better

In [13]:
banner("AUTOENCODER ARCHITECTURE TABLE")

ae_arch_df = pd.DataFrame({
    "stage": [
        "Input sequence",
        "Encoder recurrent layer",
        "Latent projection",
        "Latent expansion",
        "Repeated latent sequence",
        "Decoder recurrent layer",
        "Output projection"
    ],
    "LSTM Autoencoder": [
        "(10, n_features)",
        "LSTM(hidden_dim=64)",
        "Linear(64 → 32)",
        "Linear(32 → 64)",
        "repeat across 10 steps",
        "LSTM(hidden_dim=64)",
        "Linear(64 → n_features)"
    ],
    "GRU Autoencoder": [
        "(10, n_features)",
        "GRU(hidden_dim=64)",
        "Linear(64 → 32)",
        "Linear(32 → 64)",
        "repeat across 10 steps",
        "GRU(hidden_dim=64)",
        "Linear(64 → n_features)"
    ]
})

display(ae_arch_df)


AUTOENCODER ARCHITECTURE TABLE


,stage,LSTM Autoencoder,GRU Autoencoder
0,Input sequence,"(10, n_features)","(10, n_features)"
1,Encoder recurrent layer,LSTM(hidden_dim=64),GRU(hidden_dim=64)
2,Latent projection,Linear(64 → 32),Linear(64 → 32)
3,Latent expansion,Linear(32 → 64),Linear(32 → 64)
4,Repeated latent sequence,repeat across 10 steps,repeat across 10 steps
5,Decoder recurrent layer,LSTM(hidden_dim=64),GRU(hidden_dim=64)
6,Output projection,Linear(64 → n_features),Linear(64 → n_features)


In [14]:
banner("TEMPORAL BRANCH RESULTS")

display(nb6_metrics)

nb6_plot_df = pd.DataFrame({
    "model": nb6_metrics["model"],
    "best_val_loss": nb6_metrics["best_val_loss"],
    "test_recon_mean": nb6_metrics["test_recon_mean"],
    "test_recon_median": nb6_metrics["test_recon_median"],
    "test_latent_silhouette": nb6_metrics["test_latent_silhouette"],
    "train_time_sec": nb6_metrics["train_time_sec"]
})

nb6_long = nb6_plot_df.melt(
    id_vars=["model"],
    value_vars=["best_val_loss", "test_recon_mean", "test_recon_median", "test_latent_silhouette", "train_time_sec"],
    var_name="metric",
    value_name="value"
)

fig_nb6 = px.bar(
    nb6_long,
    x="metric",
    y="value",
    color="model",
    barmode="group",
    title="Temporal Branch — LSTM Autoencoder vs GRU Autoencoder"
)
fig_nb6.update_layout(template="plotly_white")
fig_nb6.show()

save_plot(fig_nb6, "nb8_temporal_branch_comparison")


TEMPORAL BRANCH RESULTS


,model,best_val_loss,best_epoch,train_time_sec,train_recon_mean,val_recon_mean,test_recon_mean,train_recon_median,val_recon_median,test_recon_median,test_latent_silhouette
0,LSTM,0.401665,30,1.738099,0.594315,0.401665,0.508348,0.401276,0.395802,0.452853,0.031914
1,GRU,0.390505,29,2.640879,0.572474,0.390505,0.454893,0.384876,0.388159,0.415707,0.013141


[WARN] Saved nb8_temporal_branch_comparison.html | PNG export skipped (
Image export using the "kaleido" engine requires the kaleido package,
which can be installed using pip:
    $ pip install -U kaleido
)


## Temporal branch conclusion

The temporal benchmark selected **GRU Autoencoder** as the best temporal model.

Why?
- lower validation reconstruction loss
- lower mean test reconstruction error
- good training speed

Important nuance:
- LSTM showed slightly better latent silhouette
- but both silhouette values remained low
- so reconstruction quality was the more reliable selection signal for this stage

In [15]:
banner("TEMPORAL WINNER SNAPSHOT")

temporal_snapshot = pd.DataFrame([{
    "winner": nb6_summary["best_temporal_model"],
    "window_size": nb6_summary["window_size"],
    "gru_best_val_loss": nb6_summary["gru_best_val_loss"],
    "gru_test_recon_mean": nb6_summary["gru_test_recon_mean"],
    "lstm_best_val_loss": nb6_summary["lstm_best_val_loss"],
    "lstm_test_recon_mean": nb6_summary["lstm_test_recon_mean"],
}])

display(temporal_snapshot)


TEMPORAL WINNER SNAPSHOT


,winner,window_size,gru_best_val_loss,gru_test_recon_mean,lstm_best_val_loss,lstm_test_recon_mean
0,GRU,10,0.390505,0.454893,0.401665,0.508348


# 5. Prototype methods and their architecture

## Prototype idea
The prototype branch does not train a conventional classifier.
Instead, it stores representative latent vectors and retrieves the closest historical patterns.

### Steps
1. take a sequence window
2. encode it with the selected temporal model (**GRU**)
3. obtain a latent vector
4. compare this vector to stored train latent vectors / class prototypes
5. retrieve the nearest matches
6. infer the most likely class from the nearest retrieved examples

## NearestNeighbors
A standard exact nearest-neighbor search method from scikit-learn.

### Why it is useful
- simple
- exact
- easy baseline for prototype retrieval

## FAISS
A high-performance vector search library optimized for nearest-neighbor retrieval.

### Why it is useful
- much faster search
- scalable vector retrieval
- very appropriate for prototype-based diagnosis

In [16]:
banner("PROTOTYPE BRANCH RESULTS")

display(nb7_metrics)

nb7_plot_df = nb7_metrics.copy()
nb7_long = nb7_plot_df.melt(
    id_vars=["method", "split"],
    value_vars=["accuracy", "balanced_accuracy", "macro_f1", "top3_hit_rate", "top5_hit_rate", "search_time_sec"],
    var_name="metric",
    value_name="value"
)

fig_nb7 = px.bar(
    nb7_long,
    x="metric",
    y="value",
    color="method",
    barmode="group",
    facet_col="split",
    title="Prototype Branch — NearestNeighbors vs FAISS"
)
fig_nb7.update_layout(template="plotly_white")
fig_nb7.show()

save_plot(fig_nb7, "nb8_prototype_branch_comparison")


PROTOTYPE BRANCH RESULTS


,method,split,accuracy,balanced_accuracy,macro_f1,weighted_f1,macro_precision,macro_recall,top3_hit_rate,top5_hit_rate,search_time_sec
0,NearestNeighbors,val,0.775862,0.779762,0.774184,0.773513,0.791925,0.779762,0.948276,1.000000,0.111875
1,NearestNeighbors,test,0.632353,0.632353,0.593593,0.593593,0.713986,0.632353,0.779412,0.897059,0.018781
2,FAISS,val,0.775862,0.779762,0.774184,0.773513,0.791925,0.779762,0.948276,1.000000,0.001053
3,FAISS,test,0.632353,0.632353,0.593593,0.593593,0.713986,0.632353,0.779412,0.897059,0.000530


[WARN] Saved nb8_prototype_branch_comparison.html | PNG export skipped (
Image export using the "kaleido" engine requires the kaleido package,
which can be installed using pip:
    $ pip install -U kaleido
)


## Prototype branch conclusion

The prototype benchmark selected **FAISS** as the best prototype backend.

Why?
- it matched NearestNeighbors on validation quality
- but achieved dramatically faster search times

Important interpretation:
- top-1 prototype classification is only moderate
- but top-3 and top-5 hit rates are much stronger
- so the prototype branch is more useful as a retrieval/support mechanism than as the main final classifier

In [17]:
banner("PROTOTYPE WINNER SNAPSHOT")

prototype_snapshot = pd.DataFrame([{
    "winner": nb7_summary["best_prototype_method"],
    "best_temporal_model_used": nb7_summary["best_temporal_model_used"],
    "nn_val_search_time_sec": nb7_summary["nn_val_search_time_sec"],
    "faiss_val_search_time_sec": nb7_summary["faiss_val_search_time_sec"],
    "nn_test_search_time_sec": nb7_summary["nn_test_search_time_sec"],
    "faiss_test_search_time_sec": nb7_summary["faiss_test_search_time_sec"],
}])

display(prototype_snapshot)


PROTOTYPE WINNER SNAPSHOT


,winner,best_temporal_model_used,nn_val_search_time_sec,faiss_val_search_time_sec,nn_test_search_time_sec,faiss_test_search_time_sec
0,FAISS,GRU,0.111875,0.001053,0.018781,0.00053


# 6. Final selected models

The final winners of the modeling validation are:

- **Supervised branch:** Random Forest
- **Temporal branch:** GRU Autoencoder
- **Prototype branch:** FAISS

In [18]:
banner("FINAL WINNERS TABLE")

winners_df = pd.DataFrame([
    {
        "branch": "Supervised tabular diagnosis",
        "winner": nb4_summary["best_model"],
        "why_selected": "Best corrected benchmark baseline after leakage-risk patch"
    },
    {
        "branch": "Temporal sequence representation",
        "winner": nb6_summary["best_temporal_model"],
        "why_selected": "Lower validation and test reconstruction error"
    },
    {
        "branch": "Prototype retrieval",
        "winner": nb7_summary["best_prototype_method"],
        "why_selected": "Same validation quality as NN, much faster search"
    }
])

display(winners_df)

fig_winners = px.bar(
    winners_df,
    x="branch",
    y=[1, 1, 1],
    color="winner",
    text="winner",
    title="Final Selected Winners by Modeling Branch"
)
fig_winners.update_traces(textposition="outside")
fig_winners.update_layout(
    template="plotly_white",
    yaxis_title="Selected",
    showlegend=True
)
fig_winners.show()

save_plot(fig_winners, "nb8_final_winners")


FINAL WINNERS TABLE


,branch,winner,why_selected
0,Supervised tabular diagnosis,Random Forest,Best corrected benchmark baseline after leakag...
1,Temporal sequence representation,GRU,Lower validation and test reconstruction error
2,Prototype retrieval,FAISS,"Same validation quality as NN, much faster search"


[WARN] Saved nb8_final_winners.html | PNG export skipped (
Image export using the "kaleido" engine requires the kaleido package,
which can be installed using pip:
    $ pip install -U kaleido
)


# 7. Final explanatory interpretation

## What is already validated
The current validation demonstrates that, on the current real-data benchmark:
- a strong supervised baseline can distinguish the robust benchmark classes
- a recurrent autoencoder can build a useful temporal representation
- a prototype retrieval layer can retrieve similar latent patterns efficiently


In [19]:
banner("FINAL PRESENTATION SUMMARY")

presentation_summary = pd.DataFrame([
    {
        "question": "What was the current benchmark scope?",
        "answer": "Real-data 2-class benchmark: RC_CAPACITY_OVERLOAD vs RC_TRANSPORT_DELAY"
    },
    {
        "question": "Which supervised model won?",
        "answer": nb4_summary["best_model"]
    },
    {
        "question": "Which temporal model won?",
        "answer": nb6_summary["best_temporal_model"]
    },
    {
        "question": "Which prototype method won?",
        "answer": nb7_summary["best_prototype_method"]
    },
    {
        "question": "What is the main temporal window?",
        "answer": f'{nb6_summary["window_size"]} samples'
    },
    {
        "question": "Why is the prototype branch useful?",
        "answer": "It provides fast similar-case retrieval and support explanations"
    }
])

display(presentation_summary)


FINAL PRESENTATION SUMMARY


,question,answer
0,What was the current benchmark scope?,Real-data 2-class benchmark: RC_CAPACITY_OVERL...
1,Which supervised model won?,Random Forest
2,Which temporal model won?,GRU
3,Which prototype method won?,FAISS
4,What is the main temporal window?,10 samples
5,Why is the prototype branch useful?,It provides fast similar-case retrieval and su...


# 9. Final conclusion

This modeling validation gives you a clear and defensible result:

- **Random Forest** is the best corrected supervised baseline
- **GRU Autoencoder** is the best temporal encoder
- **FAISS** is the best prototype retrieval backend

This is a strong modeling milestone for the current benchmark and provides a clean foundation for future expansion toward the broader Diagnostic Agent taxonomy.

In [20]:
banner("SAVE NOTEBOOK 8 OUTPUTS")

final_summary = {
    "benchmark_scope": {
        "target_classes": nb4_summary["target_classes"],
        "window_size": nb6_summary["window_size"],
    },
    "winners": {
        "supervised_branch": nb4_summary["best_model"],
        "temporal_branch": nb6_summary["best_temporal_model"],
        "prototype_branch": nb7_summary["best_prototype_method"],
    },
    "why_each_won": {
        "supervised_branch": "Best corrected tabular benchmark baseline",
        "temporal_branch": "Best reconstruction quality on sequence windows",
        "prototype_branch": "Same retrieval quality as NN, much faster search"
    }
}

with open("nb8_final_validation_summary.json", "w") as f:
    json.dump(final_summary, f, indent=2, default=str)

winners_df.to_csv("nb8_final_winners_table.csv", index=False)
presentation_summary.to_csv("nb8_presentation_summary_table.csv", index=False)

log("Saved Notebook 8 outputs")


SAVE NOTEBOOK 8 OUTPUTS
[INFO] Saved Notebook 8 outputs
